In [ ]:
"""
Deepseek v4.1 introduced Compressed Sparse Attention 2 (CSA2) which intends to reduce 
the kv cache storage and attention computation simultaneously.
(https://arxiv.org/pdf/2609.19969)
"""

In [ ]:
import torch
from torch import nn
import math

class SWA_KV():
    def __init__(self, window_size=32, batch=32, kv_heads=2, head_dim=128):
        self.window_size = window_size

        # B, heads, 0, head_dim
        self.k = torch.empty((batch, kv_heads, 0, head_dim))
        self.k = torch.empty((batch, kv_heads, 0, head_dim))

    def update(self, new_k, new_v):
        # k_new: B, heads, 1, head_dim
        self.k = torch.cat([self.k, new_k], dim=2)
        self.v = torch.cat([self.v, new_v], dim=2)

        # only keep the windowlength
        self.k = self.k[:, :, -self.window_size:, :]    
        self.v = self.v[:, :, -self.window_size:, :]
        
        return self.k, self.v


class CSA_2(nn.Module):
    def __init__(self, D=1024, mode="full", heads=2, d_ind = 8):
        """ 
        mode: full, reindex, reuse

        """
        super().__init__()
        assert mode in ["full", "reindex", "reuse"]
        assert D % heads == 0

        self.mode = mode
        self.head_dim = D // heads
        self.d_ind = d_ind
        self.n_heads = heads

        self.wq = nn.Linear(D, D)
        self.wk = nn.Linear(D, D)

        self.swa_kv_cache = SWA_KV(kv_heads=heads, head_dim=D)
        if self.mode == "full":
            self.k_indexer = nn.Linear(self.head_dim, d_ind)
            self.q_indexer = nn.Linear(self.head_dim, d_ind)

    def split_heads(self, x):
        # x: B, S, D
        x = x.view(x.shape[0], x.shape[1], self.n_heads, self.head_dim) # B, S, H, head_dim
        return x.transpose(1, 2)        # B, H, S, head_dim


    def forward(self, x, global_main_kv=None):
        # x: B, S, D
        q = self.wq(x)
        x_last = x[:, -1, :] # B, 1, D

        if global_main_kv==None:
            global_main_kv = torch.empty((x.shape[0], ))

        # q: B, H, S, head_dim
        q = self.split_heads(q)
        if self.mode == "full":
            # k, q indexer first
            # get k but we in the best case only need the latest S
            new_k = self.wk(x_last)         # B, 1, D
            new_k = self.split_heads(new_k)
            new_v = self.wv(x_last)
            # possibly not necessary because we first use it on the global_main_kv
            new_v = self.split_heads(new_v) # B, H, 1, h_dim
            
            k, v = self.swa_kv_cache.update(new_k, new_v) # B, :SWA, D

            # selector
            k_sel = self.k_indexer(k) 



layer = CSA_2(D=64)
x_test = torch.randn(1, 1, 64)
x = layer(x_test)
x

TypeError: SWA_KV.update() missing 1 required positional argument: 'new_v'

In [ ]:
"""
What is an indexer?
"""
x = torch.randn((5, 10)) # t, d
kv = torch.randn((20, 10)) # s, d
# we need t,d -> s actually 
# (t, d), (s,d) -> s are they always padded?!
# bambus 
kv_indexer = torch.nn.Linear(5, 30) # t,d -> s 
# the most bambus solution 
idx = kv_indexer(x.T) # d, t -> d, s
idx = torch.mean(idx, dim=0) # d, s -> s
idx.shape # hell yeah



torch.Size([30])